# 04 — R5: adverse-action reason codes and whether they hold still

**ADIL** · MAIB AI 217 (AI in Finance) · SP Jain School of Global Management, Dubai · Krishna Mathur

R5 is not a model. It is a **gate**, and it asks the question a regulator actually asks: show
me a declined applicant's reasons, and show me they mean something.

An explanation that changes when the applicant's file is nudged is not an explanation. So the
test is: compute each declined applicant's top-3 reasons, perturb their file, recompute, and
count how many are now told to fix something different.

The threshold was registered in `adil.reasons` **before any flip rate was measured** and is not
revised here. That discipline is the whole point — a gate tuned until the model passes is not
a gate, and the tuning would be invisible in the report.

The scorecard is measured on exactly the same terms. For a linear model on
weight-of-evidence inputs the SHAP value of a characteristic is *exactly* its coefficient
times its centred WoE, so R0 and the challenger are compared like for like rather than by
analogy.

Outputs `reports/explainability.md`, `metrics/r5.json`, and the reason codes themselves.

In [ ]:
import json
import pickle
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import shap

from adil import challenger, paths, reasons
from adil import split as sp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)

processed = paths.processed_dir()
frame = pd.read_parquet(processed / "adil_frame.parquet")
splits = pd.read_parquet(processed / "split_index.parquet")["split"].values
manifest = pd.read_parquet(processed / "feature_manifest.parquet").set_index("feature")
challenger_predictions = pd.read_parquet(processed / "challenger_predictions.parquet")
r0_predictions = pd.read_parquet(processed / "r0_predictions.parquet")
is_test = splits == "test"

models, model_features = {}, {}
for name in ("R1", "R3", "R4"):
    models[name] = lgb.Booster(model_file=str(processed / f"{name.lower()}_model.txt"))
    model_features[name] = pd.read_parquet(processed / f"{name.lower()}_features.parquet")[
        "feature"
    ].tolist()

with (processed / "r0_scorecard.pkl").open("rb") as handle:
    bundle = pickle.load(handle)
card, characteristics = bundle["scorecard"], bundle["characteristics"]

print(reasons.GATE_REGISTRATION)

## 1. Verification — the contributions are TreeSHAP

LightGBM computes exact TreeSHAP internally through `pred_contrib=True`, which is far faster
than routing 567 features through the `shap` package for every perturbation on the curve
below. Using it is only legitimate if it agrees with the reference implementation, so it is
checked against `shap.TreeExplainer` rather than assumed.

In [ ]:
sample = frame.loc[is_test].iloc[:300]
design = challenger.design_matrix(sample, model_features["R4"])

internal = models["R4"].predict(design, pred_contrib=True)
explainer = shap.TreeExplainer(models["R4"])
reference = explainer.shap_values(design)
reference = reference[1] if isinstance(reference, list) else reference

print(f"contributions checked on {len(sample)} applications, {len(model_features['R4'])} features")
print(
    f"max |internal - shap.TreeExplainer| : "
    f"{np.abs(np.asarray(reference) - internal[:, :-1]).max():.3e}"
)
print(
    f"base value internal / reference     : {internal[0, -1]:.10f} / "
    f"{float(np.ravel(explainer.expected_value)[0]):.10f}"
)

## 2. Contributions for both model classes

The challenger's come from TreeSHAP. The scorecard's are computed from the closed form: for a
linear model, a feature's SHAP value is its coefficient times its centred input, and the
scorecard's inputs are WoE-transformed characteristics. Both are negated so that a **positive
contribution always means "this pushed the applicant's default risk up"**, which is what makes
a feature an adverse-action reason in the first place.

In [ ]:
binning = card.binning_process_
coefficients = card.estimator_.coef_.ravel()
train_woe = pd.DataFrame(
    binning.transform(frame.loc[splits == "train", characteristics], metric="woe"),
    columns=characteristics,
)
woe_centre = train_woe.values.mean(axis=0)


def scorecard_contributions(applications):
    transformed = pd.DataFrame(
        binning.transform(applications[characteristics], metric="woe"), columns=characteristics
    )
    # The scorecard's logistic regression predicts the *good* outcome on the WoE scale,
    # so the sign is flipped to put default risk on the positive side, matching SHAP.
    return -(transformed.values - woe_centre) * coefficients


def challenger_contributions(rung, applications):
    columns = model_features[rung]
    contributions = models[rung].predict(
        challenger.design_matrix(applications, columns), pred_contrib=True
    )
    return contributions[:, :-1]


CONTRIBUTORS = {
    "R0": (lambda rows: scorecard_contributions(rows), characteristics),
    "R1": (lambda rows: challenger_contributions("R1", rows), model_features["R1"]),
    "R3": (lambda rows: challenger_contributions("R3", rows), model_features["R3"]),
    "R4": (lambda rows: challenger_contributions("R4", rows), model_features["R4"]),
}
print({name: len(cols) for name, (_, cols) in CONTRIBUTORS.items()})

## 3. Who is declined

The operative approval cutoff is set in notebook 06, from a cost matrix. It cannot be borrowed
here without making the two notebooks circular, so this one works with the **highest-risk
decile** of the test split under each model.

That population contains the declines under any plausible cutoff, and the property being
measured — whether an explanation holds still — belongs to the explanation method, not to
where the cutoff happens to sit. Each model declines a different tenth of applicants, which is
correct: they disagree about who is risky.

In [ ]:
PROBABILITY_COLUMN = {
    "R0": ("r0", "prob_calibrated"),
    "R1": ("challenger", "R1_prob_calibrated"),
    "R3": ("challenger", "R3_prob_calibrated"),
    "R4": ("challenger", "R4_prob_calibrated"),
}

declined = {}
for name, (source, column) in PROBABILITY_COLUMN.items():
    table = r0_predictions if source == "r0" else challenger_predictions
    risk = table.loc[is_test, column].values
    declined[name] = np.flatnonzero(is_test)[risk >= np.quantile(risk, 0.90)]

pd.DataFrame(
    [
        {
            "rung": name,
            "declined": len(index),
            "min probability": float(
                (r0_predictions if PROBABILITY_COLUMN[name][0] == "r0" else challenger_predictions)
                .iloc[index][PROBABILITY_COLUMN[name][1]]
                .min()
            ),
        }
        for name, index in declined.items()
    ]
).set_index("rung").round(4)

## 4. What a declined applicant is actually told

A raw column name is not a reason. `BUR_DAYS_CREDIT_MAX` tells an applicant nothing, and an
adverse-action notice built from it would satisfy a checkbox and no one else.

The labels below are composed from the feature manifest and Home Credit's own published column
descriptions. They are a demonstration that the mapping is mechanically possible and traceable,
**not** production reason text — real notices need wording authored and signed off by the
lender, and some of these features could not responsibly be put in front of a customer at all.

In [ ]:
descriptions = pd.read_csv(
    paths.home_credit_dir() / "HomeCredit_columns_description.csv", encoding="latin-1"
)
description_by_column = (
    descriptions.drop_duplicates("Row").set_index("Row")["Description"].to_dict()
)

AGGREGATION_WORDS = {
    "MIN": "lowest",
    "MAX": "highest",
    "MEAN": "average",
    "SUM": "total",
    "COUNT": "number of",
    "SHARE": "share of",
}


def readable(feature):
    if feature not in manifest.index:
        return feature
    source_column = manifest.loc[feature, "source_column"]
    aggregation = str(manifest.loc[feature, "aggregation"]).split()[0]
    text = description_by_column.get(source_column, source_column)
    text = str(text).strip().rstrip(".")
    if aggregation in AGGREGATION_WORDS and not feature.endswith("_SENTINEL_SHARE"):
        return f"{AGGREGATION_WORDS[aggregation]}: {text}"
    if feature.endswith("_SENTINEL_SHARE"):
        return f"share of prior applications with no recorded value for: {text}"
    return text


pd.DataFrame(
    {"feature": model_features["R4"], "reason text": [readable(f) for f in model_features["R4"]]}
).head(20)

In [ ]:
contributions_r4 = CONTRIBUTORS["R4"][0](frame.iloc[declined["R4"]])
reason_codes = reasons.reason_frame(
    frame.iloc[declined["R4"]]["SK_ID_CURR"].values, contributions_r4, model_features["R4"]
)

print(f"reason codes for {len(reason_codes):,} declined applicants under R4")
print(
    f"applicants with fewer than {reasons.TOP_K} reasons: "
    f"{int((reason_codes['n_reasons'] < reasons.TOP_K).sum())}"
)
print("")
print("most frequent first reason:")
print(reason_codes["reason_1"].value_counts().head(8).to_string())

In [ ]:
for position in range(3):
    row = reason_codes.iloc[position]
    probability = challenger_predictions.iloc[declined["R4"][position]]["R4_prob_calibrated"]
    print(f"application {int(row['SK_ID_CURR'])}  predicted default probability {probability:.3f}")
    for slot in range(1, reasons.TOP_K + 1):
        feature = row[f"reason_{slot}"]
        if feature:
            print(f"   {slot}. {readable(feature)}")
    print("")

## 5. Do the two model classes agree about *why*?

Both models decline people. A more searching question is whether they decline them for the
same reasons. Measured on the applicants both place in their highest-risk decile.

In [ ]:
shared_applicants = np.intersect1d(declined["R0"], declined["R4"])
shared_frame = frame.loc[shared_applicants]

r0_reasons = reasons.top_reasons(CONTRIBUTORS["R0"][0](shared_frame), characteristics)
r4_reasons = reasons.top_reasons(CONTRIBUTORS["R4"][0](shared_frame), model_features["R4"])

overlap = [len(set(a) & set(b)) for a, b in zip(r0_reasons, r4_reasons, strict=True)]
top_agree = [bool(a and b and a[0] == b[0]) for a, b in zip(r0_reasons, r4_reasons, strict=True)]
shared_features = set(characteristics) & set(model_features["R4"])

print(f"applicants in both models' highest-risk decile: {len(shared_applicants):,}")
print(f"characteristics available to both models       : {len(shared_features)}")
print(f"identical top reason                           : {np.mean(top_agree):.1%}")
print(f"at least one reason in common                  : {np.mean(np.array(overlap) > 0):.1%}")
print(f"mean shared reasons out of {reasons.TOP_K}                   : {np.mean(overlap):.2f}")

## 6. The stability curve

Every numeric feature in an applicant's file is perturbed by an independent normal draw scaled
to that feature's own standard deviation, and the reasons are recomputed.

The registered gate reads this curve at a single point. The curve itself is reported because
one reading is a verdict and the shape is an argument, and — as it turns out below — the shape
is where the interesting behaviour is.

In [ ]:
SIGMAS = [0.01, 0.05, 0.10, 0.25, reasons.PERTURBATION_SIGMA]

curve = []
for name, (contribute, columns) in CONTRIBUTORS.items():
    applications = frame.iloc[declined[name]]
    baseline = reasons.top_reasons(contribute(applications), columns)
    row = {"rung": name, "features": len(columns), "declined": len(applications)}
    for sigma in SIGMAS:
        moved = reasons.perturb(applications, columns, sigma=sigma, seed=sp.SEED)
        row[sigma] = reasons.flip_rate(baseline, reasons.top_reasons(contribute(moved), columns))
    curve.append(row)

stability = pd.DataFrame(curve).set_index("rung")
stability.round(3)

In [ ]:
registered = stability[reasons.PERTURBATION_SIGMA]
verdicts = pd.DataFrame(
    {
        "flip rate at registered sigma": registered,
        "registered gate": reasons.FLIP_RATE_GATE,
        "verdict": [reasons.gate_verdict(value) for value in registered],
    }
)
print(
    f"R5 gate: at most {reasons.FLIP_RATE_GATE:.0%} of declined applicants may see their "
    f"top-{reasons.TOP_K} reason set change"
)
print(f"at a perturbation of {reasons.PERTURBATION_SIGMA} standard deviations per feature.")
print("")
verdicts.round(4)

### The result, and what it does and does not say

**Every model fails, the scorecard included.** That is the finding, and the registered
threshold stands exactly as written.

Two things follow, and they pull in different directions.

The first is about the models. Read at small perturbations, the two classes are clearly
different: the scorecard's reasons are roughly twice as stable as the best challenger's, and
R4's are markedly more stable than R1's. Feature count is the driver — with 567 features
competing for three slots, small contribution changes reshuffle the ranking constantly.
Restricting the challenger to 20 features bought explanation stability, which is a benefit of
rung R4 that nothing in notebook 03's metrics could show.

The second is about the gate, and it is a criticism of my own registration. At half a standard
deviation on **every** numeric field at once, no model of either class survives — so the gate
as registered does not discriminate between them. Registering a threshold without a pilot
measurement produced a test that is honest but uninformative at its chosen operating point.
The right response is to report that plainly, not to move the threshold: moving it after
seeing the numbers is precisely the failure the registration exists to prevent.

The curves also **cross**. At the registered perturbation the scorecard is *worse* than R4.
That is not noise, and the mechanism is legible: a scorecard's contribution changes only when a
perturbation pushes a characteristic across a bin boundary, and at half a standard deviation
almost every characteristic crosses one, so the whole points table reshuffles at once. The
gradient booster's contributions vary more smoothly. Coarse binning buys stability against
small nudges and gives it back against large ones.

The honest headline for the research question: **SHAP reason codes on a gradient booster are
not stable enough to serve as adverse-action reasons at any perturbation scale tested, and
neither are a scorecard's beyond very small ones.** Reason-code stability is a real constraint
that neither model class satisfies, and it is not visible in any discrimination metric.

## 7. Persist

In [ ]:
reason_codes["reason_1_text"] = [readable(f) if f else "" for f in reason_codes["reason_1"]]
reason_codes.to_parquet(processed / "reason_codes.parquet", index=False)
stability.to_parquet(processed / "reason_stability.parquet")

payload = {
    "rung": "R5",
    "kind": "gate — pass/fail, changes no model",
    "seed": sp.SEED,
    "registration": reasons.GATE_REGISTRATION,
    "top_k": reasons.TOP_K,
    "registered_sigma": reasons.PERTURBATION_SIGMA,
    "registered_gate": reasons.FLIP_RATE_GATE,
    "declined_population": "highest-risk decile of the test split, per model",
    "verification": {
        "treeshap_vs_shap_library_max_abs_diff": float(
            np.abs(np.asarray(reference) - internal[:, :-1]).max()
        )
    },
    "flip_rate_curve": {
        name: {str(sigma): float(stability.loc[name, sigma]) for sigma in SIGMAS}
        for name in stability.index
    },
    "verdicts": {
        name: {
            "flip_rate": float(registered[name]),
            "verdict": reasons.gate_verdict(float(registered[name])),
        }
        for name in stability.index
    },
    "cross_model_agreement": {
        "shared_applicants": int(len(shared_applicants)),
        "identical_top_reason": float(np.mean(top_agree)),
        "at_least_one_shared_reason": float(np.mean(np.array(overlap) > 0)),
        "mean_shared_reasons": float(np.mean(overlap)),
    },
}
(paths.metrics_dir() / "r5.json").write_text(json.dumps(payload, indent=2, default=float) + "\n")
print("wrote metrics/r5.json")

In [ ]:
smallest = SIGMAS[0]
r0_small, r4_small = stability.loc["R0", smallest], stability.loc["R4", smallest]
r1_small = stability.loc["R1", smallest]
r0_big = stability.loc["R0", reasons.PERTURBATION_SIGMA]
r4_big = stability.loc["R4", reasons.PERTURBATION_SIGMA]

lines = [
    "# ADIL — R5: adverse-action reason codes and whether they hold still",
    "",
    "Generated by `notebooks/04_explainability.ipynb`. Every number is computed, not typed.",
    "",
    "MAIB AI 217 · SP Jain School of Global Management, Dubai · Krishna Mathur",
    "",
    "## The gate, as registered",
    "",
    "> " + reasons.GATE_REGISTRATION,
    "",
    "Registered in `adil.reasons` before any flip rate was measured, and not revised.",
    "",
    "## Verification",
    "",
    "Contributions come from LightGBM's internal TreeSHAP (`pred_contrib=True`), which is",
    "fast enough to recompute across the whole perturbation curve. It is checked against",
    "`shap.TreeExplainer` rather than assumed:",
    "",
    f"- Maximum absolute difference over {len(sample)} applications x "
    f"{len(model_features['R4'])} features: "
    f"**{np.abs(np.asarray(reference) - internal[:, :-1]).max():.1e}**",
    "",
    "The scorecard's contributions are not an analogy. For a linear model a feature's SHAP",
    "value is exactly its coefficient times its centred input, and the scorecard is linear on",
    "weight-of-evidence inputs, so R0 and the challenger are compared on identical terms.",
    "",
    "## Verdict",
    "",
    f"| Rung | Features | Declined | Flip rate at {reasons.PERTURBATION_SIGMA}σ | Gate | Verdict |",
    "|---|---:|---:|---:|---:|---|",
]
for name in ["R0", "R1", "R3", "R4"]:
    lines.append(
        f"| {name} | {int(stability.loc[name, 'features'])} | "
        f"{int(stability.loc[name, 'declined']):,} | "
        f"{registered[name]:.3f} | {reasons.FLIP_RATE_GATE:.2f} | "
        f"**{reasons.gate_verdict(float(registered[name]))}** |"
    )
lines += [
    "",
    "**Every model fails, the scorecard included.** The registered threshold stands as",
    "written.",
    "",
    "## The stability curve",
    "",
    "Share of declined applicants whose top-3 reason set changes, by perturbation scale.",
    "Order within the set does not count as a change; substitution does.",
    "",
    "| Rung | Features | " + " | ".join(f"{s}σ" for s in SIGMAS) + " |",
    "|---|---:|" + "---:|" * len(SIGMAS),
]
for name in ["R0", "R1", "R3", "R4"]:
    cells_out = " | ".join(f"{stability.loc[name, s]:.3f}" for s in SIGMAS)
    lines.append(f"| {name} | {int(stability.loc[name, 'features'])} | {cells_out} |")
lines += [
    "",
    "### What this says",
    "",
    "**Feature count drives explanation stability.** At the smallest perturbation tested",
    f"({smallest}σ) the scorecard flips {r0_small:.1%} of reason sets, R4 flips",
    f"{r4_small:.1%}, and the unconstrained R1 flips {r1_small:.1%}. With 567 features",
    "competing for three slots, small changes in contribution reshuffle the ranking",
    "constantly. Restricting the challenger to 20 features bought explanation stability —",
    "a benefit of rung R4 that no discrimination metric in `challenger.md` can show, and one",
    "that partly offsets the PR-AUC that rung gave up.",
    "",
    "**The registered gate does not discriminate, and that is a fault in the registration.**",
    "At half a standard deviation applied to every numeric field at once, no model of either",
    "class survives. Registering a threshold without a pilot measurement produced a test that",
    "is honest but uninformative at its operating point. The correct response is to report",
    "that, not to move the threshold: moving it after seeing the numbers is exactly the",
    "failure the registration exists to prevent. A future iteration should register at a",
    "scale representing plausible data error rather than a materially different applicant.",
    "",
    f"**The curves cross.** At the registered scale the scorecard ({r0_big:.3f}) is *worse*",
    f"than R4 ({r4_big:.3f}). The mechanism is legible rather than noise: a scorecard's",
    "contribution changes only when a perturbation pushes a characteristic over a bin",
    "boundary, and at half a standard deviation nearly every characteristic crosses one, so",
    "the points table reshuffles wholesale. The booster's contributions vary more smoothly.",
    "Coarse binning buys stability against small nudges and surrenders it against large ones.",
    "",
    "**The headline, stated plainly.** SHAP reason codes on a gradient booster are not stable",
    "enough to serve as adverse-action reasons at any perturbation scale tested here, and a",
    "scorecard's are stable only under very small ones. Reason-code stability is a real",
    "constraint that neither model class satisfies, and it is invisible in every",
    "discrimination metric this project reports.",
    "",
    "## Do the two model classes agree about why?",
    "",
    f"Measured on the {len(shared_applicants):,} applicants both R0 and R4 place in their",
    f"highest-risk decile. {len(shared_features)} characteristics are available to both.",
    "",
    f"- Identical top reason: **{np.mean(top_agree):.1%}**",
    f"- At least one reason in common: **{np.mean(np.array(overlap) > 0):.1%}**",
    f"- Mean shared reasons out of {reasons.TOP_K}: **{np.mean(overlap):.2f}**",
    "",
    "Two models can agree closely on *who* is risky and still disagree on *why*. Only the",
    "second is what an applicant is told.",
    "",
    "## Reason text",
    "",
    "A raw column name is not a reason. `BUR_DAYS_CREDIT_MAX` tells an applicant nothing.",
    "Labels are composed from the feature manifest and Home Credit's published column",
    "descriptions, which demonstrates the mapping is mechanical and traceable. It is **not**",
    "production reason text: real notices need wording authored and signed off by the lender,",
    "and some of these features could not responsibly be shown to a customer at all.",
    "",
    "Most frequent first reason under R4:",
    "",
    "| Feature | Reason text | Applicants |",
    "|---|---|---:|",
]
for feature, count in reason_codes["reason_1"].value_counts().head(8).items():
    lines.append(f"| `{feature}` | {readable(feature)} | {int(count):,} |")
lines += [
    "",
    "## Limitations",
    "",
    "- The declined population is the highest-risk decile per model, not the cutoff notebook",
    "  06 sets from a cost matrix. Using the operative cutoff here would make the two",
    "  notebooks circular; explanation stability is a property of the method rather than of",
    "  where the cutoff sits.",
    "- Perturbation moves every numeric field independently and simultaneously, which is a",
    "  harsher test than a single mis-keyed value and is not a model of any real data error",
    "  process.",
    "- Categorical features are not perturbed. There is no scale on which to move a category",
    "  by a fraction of a standard deviation, so their contribution to instability is",
    "  unmeasured and the flip rates below are, in that respect, optimistic.",
    "- SHAP is locally faithful to the model, not to the world. A stable reason code is not",
    "  thereby a correct or an actionable one.",
    "",
]
path = paths.reports_dir() / "explainability.md"
path.write_text("\n".join(lines) + "\n")
print(f"wrote {path} ({len(lines)} lines)")